# PS66 — Train on Google Colab

Satellite-embedding CNN for 3-D subsurface ocean temperature (North Indian Ocean, 2020).

**Plan**
1. Mount Google Drive
2. Load the project + prebuilt `X`/`Y` arrays
3. Verify the data & GPU
4. Train the encoder–decoder
5. Save checkpoint + figures back to Drive

**Put these on Drive first (folder `MyDrive/ps66/`):**
- `siH-PS66.zip` — this repository (zipped)
- `arrays/` — `X_2020.npy`, `Y_2020.npy`, `stats_2020.npz`, `channels_2020.npy`, `bathy_2020.npy`

> Arrays are produced locally with:
> `py -3.12 build_from_local.py --dir ".../Desktop/model" --tag 2020`


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Load the repo + arrays

In [ ]:
import os, glob, shutil, zipfile, pathlib

BASE = "/content/drive/MyDrive/ps66"      # <-- edit if your folder differs
assert os.path.exists(BASE), f"not found: {BASE}"

# --- unzip the repo ---
if os.path.exists("/content/_repo"):
    shutil.rmtree("/content/_repo")
os.makedirs("/content/_repo", exist_ok=True)
with zipfile.ZipFile(f"{BASE}/siH-PS66.zip") as z:
    z.extractall("/content/_repo")

# find the folder that actually contains run_day3.py
hits = list(pathlib.Path("/content/_repo").glob("**/run_day3.py"))
assert hits, "run_day3.py not found in the zip"
REPO = str(hits[0].parent)
print("REPO:", REPO)

# --- copy the prebuilt arrays into data/processed ---
DATA = os.path.join(REPO, "data", "processed")
os.makedirs(DATA, exist_ok=True)
arr = glob.glob(f"{BASE}/arrays/*")
assert arr, f"no array files in {BASE}/arrays"
for f in arr:
    shutil.copy(f, DATA)
print("copied:", sorted(os.path.basename(f) for f in arr))

os.chdir(REPO)
print("cwd:", os.getcwd())

## 3. Verify environment, GPU and data

In [ ]:
import tensorflow as tf
print("tensorflow", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

import numpy as np
X = np.load("data/processed/X_2020.npy", mmap_mode="r")
Y = np.load("data/processed/Y_2020.npy", mmap_mode="r")
print("X", X.shape, " Y", Y.shape)
print("channels:", list(np.load("data/processed/channels_2020.npy", allow_pickle=True)))
print("days:", X.shape[0], " valid@0m: %.1f%%" % (np.isfinite(np.asarray(Y[:,0])).mean()*100))

## 4. (Optional) Build arrays on Colab from raw NetCDFs

Skip this if you already uploaded `arrays/`. Only needed if you'd rather put the
raw `.nc` files on Drive (folder `MyDrive/ps66/raw/`) and build here.

> Colab free tier has ~12 GB RAM; the full-year build peaks around 10 GB.
> If it gets killed, build locally instead.

In [ ]:
# from src import harmonize, config
# from pathlib import Path
# import numpy as np
#
# RAW = Path("/content/drive/MyDrive/ps66/raw")
# def find(*needles):
#     hits = [p for p in RAW.rglob("*.nc") if all(n.lower() in p.name.lower() for n in needles)]
#     hits.sort(key=lambda p: p.stat().st_size, reverse=True)
#     return hits[0]
#
# out = harmonize.build_arrays(
#     find("phy-all_my_0.25deg", "thetao_glor"),
#     find("duacs"),
#     find("glo-sst"),
#     sss_path=find("sss_my_multi"),
#     tag="2020",
# )
# print("X", out["X"].shape, "Y", out["Y"].shape)

## 5. Train + evaluate

Trains the encoder–decoder, evaluates on the last 20 % of days (common mask),
prints the honest headline vs persistence, and writes a RMSE-vs-depth figure.

In [ ]:
%run run_day3.py 2020 120

## 6. Save checkpoint + figures to Drive

In [ ]:
import shutil, os, glob
DST = f"{BASE}/outputs"
os.makedirs(DST, exist_ok=True)

for p in glob.glob("checkpoints/*.weights.h5"):
    shutil.copy(p, DST)
for p in glob.glob("figures/*.png"):
    shutil.copy(p, DST)
print("saved:", sorted(os.path.basename(p) for p in glob.glob(DST + "/*")))

## 7. Inspect the result

`run_day3.py` saves `checkpoints/model_2020.weights.h5`. To reload it later:

```python
from src import model as M
m = M.OceanModel(cin=5, D=256, Z=15)
m(tf.zeros([1,5,101,241]))
m.load_weights("/content/drive/MyDrive/ps66/outputs/model_2020.weights.h5")
```

| Artifact | Meaning |
|---|---|
| `model_2020.weights.h5` | trained weights |
| `model_rmse_2020.png` | RMSE vs depth, model + baselines |